
# MedLLaMA Clinical Summarization (Text-Only) + ROUGE Eval

This notebook reproduces the *workflow* of your LLaVA‑Med summary pipeline but swaps in a MedLLaMA model for **text-only** summarization.  
It expects a CSV with columns:
- `Preprocessed Posts` — the input text to summarize
- `generated_summary` — a gold/reference summary for evaluation (optional but used for ROUGE)

It supports 4‑bit loading (bitsandbytes) for VRAM efficiency and produces a readable results file plus aggregate ROUGE metrics.

> Adapted to mirror the structure and I/O style of your existing script.  
> Fill in the model ID for your preferred **MedLLaMA** checkpoint (Hugging Face) and file paths, then run.


In [ ]:

# If needed, uncomment these:
# !pip install -U transformers accelerate bitsandbytes torch rouge-score
# For CUDA-enabled PyTorch, please ensure a compatible torch is installed.


In [ ]:

import os, time
from typing import Optional, List, Tuple
import torch
import pandas as pd

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from rouge_score import rouge_scorer, scoring

# --- GPU sanity ---
assert torch.cuda.is_available(), "Please enable a GPU runtime (CUDA required for reasonable speed)."
print("CUDA device:", torch.cuda.get_device_name(0))
torch.backends.cuda.matmul.allow_tf32 = True


In [ ]:

# ==== CONFIG ====
# 👉 Provide a valid MedLLaMA-ish checkpoint here (HF Hub). Examples you might try:
# - 'medllama/medllama-7b' (placeholder — replace with the exact repo you use)
# - 'microsoft/Orca-MedLM-7b' (example of a medical-tuned LLM; change if you have a specific MedLLaMA)
# - any compatible HF Causal LM you use internally as "MedLLaMA"
MODEL_ID = "medllama/medllama-7b"  # <-- TODO: set to your exact model repo ID

# Input/Output paths (change to your actual files)
CSV_PATH = "./data/LLaVA-Med/file3.csv"  # expects 'Preprocessed Posts' & 'generated_summary'
READABLE_OUT = "./data/LLaVA-Med/Results_MedLLaMA_Summary_11_11_1.txt"

# Limit how many rows to process (None to process all)
MAX_ROWS: Optional[int] = 50

# Generation params
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.0       # deterministic while benchmarking
TOP_P = 1.0
DO_SAMPLE = False

# 4-bit quantization to fit on common GPUs. Set to False for full precision.
USE_4BIT = True


In [ ]:

bnb_4bit = None
if USE_4BIT:
    bnb_4bit = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

print("Loading model:", MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16 if not USE_4BIT else None,
    quantization_config=bnb_4bit,
    low_cpu_mem_usage=True,
)
model.eval()
print("Model loaded.")


In [ ]:

SYSTEM_PROMPT = (
    "You are an excellent clinical information extraction system. Summarise the adverse drug events and drugs causing that if mentioned in the provided text. An adverse drug event is any harmful or negative experience related to ongoing medication or treatment."
)

def build_prompt(user_text: str) -> str:
    # Many MedLLaMA checkpoints use Llama-style chat templates; here we keep it generic and robust.
    # If your tokenizer exposes apply_chat_template, prefer it.
    if hasattr(tokenizer, "apply_chat_template"):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_text},
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # Fallback to a simple instruct format
    return (
        "<<SYS>>\n" + SYSTEM_PROMPT + "\n<</SYS>>\n"
        "[INST] Summarize the following text.\n\n" + user_text + " [/INST]"
    )

@torch.inference_mode()
def generate_summary(user_text: str, max_new_tokens: int = 256) -> str:
    prompt = build_prompt(user_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=DO_SAMPLE,
        temperature=TEMPERATURE if DO_SAMPLE else None,
        top_p=TOP_P if DO_SAMPLE else None,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    # Heuristic: if the template echos the prompt, strip it.
    if "[/INST]" in text:
        text = text.split("[/INST]", 1)[-1].strip()
    return text.strip()


In [ ]:

df = pd.read_csv(CSV_PATH)
if MAX_ROWS is not None:
    df = df.head(MAX_ROWS).copy()

assert 'Preprocessed Posts' in df.columns, "CSV must contain a 'Preprocessed Posts' column."
has_reference = 'generated_summary' in df.columns

results_lines = []
start = time.perf_counter()

header = f"MedLLaMA Summarization — MODEL_ID={MODEL_ID}  rows={len(df)}"
print(header)
results_lines.append(header)

aggregator = scoring.BootstrapAggregator() if has_reference else None
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True) if has_reference else None

for idx, row in df.iterrows():
    context = str(row['Preprocessed Posts'])
    # Build user query aligned with prior pipeline
    query = (
        "Summarise the adverse drug events and the drugs causing them (if mentioned) "
        "in the provided text. An adverse drug event is any harmful or negative experience "
        "related to ongoing medication or treatment.\n\nText: "
        + context
    )

    print("\n" + "-"*70)
    results_lines.append("\n" + "-"*70)
    print(f"🧠 Context: {context}")
    results_lines.append(f"🧠 Context: {context}")

    if has_reference:
        ref = str(row['generated_summary'])
        print(f"🪙 Gold Reference: {ref}")
        results_lines.append(f"🪙 Gold Reference: {ref}")

    pred = generate_summary(query, max_new_tokens=MAX_NEW_TOKENS)
    print(f"💬 MedLLaMA Summary: {pred}")
    results_lines.append(f"💬 MedLLaMA Summary: {pred}")

    if has_reference:
        scores = scorer.score(ref, pred)
        if aggregator is not None:
            aggregator.add_scores(scores)
        for metric, s in scores.items():
            line = f"   {metric.upper():7s} | P={s.precision:.3f}  R={s.recall:.3f}  F1={s.fmeasure:.3f}"
            print(line)
            results_lines.append(line)

print("\n" + "="*70)
results_lines.append("\n" + "="*70)

if has_reference and aggregator is not None:
    aggregated = aggregator.aggregate()
    print("🔹 Final Aggregate (bootstrap mid) ROUGE scores:")
    results_lines.append("🔹 Final Aggregate (bootstrap mid) ROUGE scores:")
    for metric, agg in aggregated.items():
        mid = agg.mid
        line = f"{metric.upper():7s} | P={mid.precision:.3f}  R={mid.recall:.3f}  F1={mid.fmeasure:.3f}"
        print(line)
        results_lines.append(line)

    final_avg_f1 = sum(aggregated[m].mid.fmeasure for m in aggregated) / len(aggregated)
    print("-"*70)
    results_lines.append("-"*70)
    print(f"⭐ Final Average F1 (ROUGE-1, ROUGE-2, ROUGE-L): {final_avg_f1:.3f}")
    results_lines.append(f"⭐ Final Average F1 (ROUGE-1, ROUGE-2, ROUGE-L): {final_avg_f1:.3f}")
else:
    print("No 'generated_summary' column found — skipping ROUGE evaluation.")
    results_lines.append("No 'generated_summary' column found — skipping ROUGE evaluation.")

elapsed = time.perf_counter() - start
print("-"*70)
print(f"Execution time: {elapsed:.2f} s")
results_lines.append("-"*70)
results_lines.append(f"Execution time: {elapsed:.2f} s")

# Write human-readable log
os.makedirs(os.path.dirname(READABLE_OUT), exist_ok=True)
with open(READABLE_OUT, "w", encoding="utf-8") as f:
    f.write("\n".join(results_lines))

print(f"\n--- Readable results written to: {READABLE_OUT} ---")


In [ ]:

# 🔎 Quick test on an ad-hoc snippet
sample_text = """
Patient reported dizziness and nausea after initiating amlodipine. Symptoms improved on dose reduction,
but peripheral edema persisted. No prior history of vestibular disorders.
"""
print(generate_summary("Summarise ADEs and implicated drugs.\n\nText: " + sample_text))
